## Update Yearly Like playlists wit Musicbee likes

In [1]:
import time

import pandas as pd
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)

from ytmusic_library import YTMusicPlaylists

DATE = time.strftime('%m-%d-%Y')

HEADER_FILE = '../oauth.json'
PLAYLIST_TSV_DIR = '../playlists/'
Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)

uni_module_path = os.path.abspath(os.path.join('../../music-sources-unified'))
if uni_module_path not in sys.path: sys.path.append(uni_module_path)
import unify_lib as uni

# MB_LIB = os.path.join(module_path, 'db_assets/musicbee_library.tsv')
# MB_INBOX = os.path.join(module_path, 'db_assets/musicbee_inbox.tsv')

# print('Loading ytmusic and musicbee track databases (takes ~1m)')
# mb_tracks = uni.ingest_musicbee_db_assets(
#     MB_LIB, MB_INBOX, save_tsv=False
# )

# YT_TRACK_DB = os.path.join(PLAYLIST_TSV_DIR, '_tracks_db.tsv')
# yt_tracks = uni.ingest_ytmusic_db_assets( YT_TRACK_DB, save_tsv=False)

ALBUM_TRACK_MATCH_TSV = os.path.join(
    uni_module_path, 'tsvs', 'musicbee_track_matches_for_yt_album_matches.tsv'
)

ARTIST_TRACK_MATCH_TSV = os.path.join(
    uni_module_path, 'tsvs', 'musicbee_track_matches_for_yt_artist_matches.tsv'
)
artist_track_matches = pd.read_csv(ARTIST_TRACK_MATCH_TSV, sep='\t')
album_track_matches = pd.read_csv(ALBUM_TRACK_MATCH_TSV, sep='\t')
artist_track_matches = artist_track_matches.loc[
    artist_track_matches['mb_match_label'] == 'MATCH'
]
album_track_matches = album_track_matches.loc[
    album_track_matches['mb_match_label'] == 'MATCH'
]

# Overwrite artist matches with album matches if same path
mb_path_map = dict(zip(artist_track_matches['mb_Path'], artist_track_matches['yt_videoId']))
mb_path_map.update(dict(zip(album_track_matches['mb_Path'], album_track_matches['yt_videoId']))) 

print(f'{len(mb_path_map)} yt_videoIds have a mb path')


Using header file: ../oauth.json
50302 yt_videoIds have a mb path


C:\Users\jake\AppData\Local\Temp\ipykernel_14624\2987992531.py:40: DtypeWarning: Columns (25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  album_track_matches = pd.read_csv(ALBUM_TRACK_MATCH_TSV, sep='\t')


In [5]:

# PLAYLIST_NAME = 'zz not like'
# playlist_file = os.path.join(PLAYLIST_TSV_DIR, PLAYLIST_NAME + '.tsv')
directory = 'D:\\Music\\MusicBee\\mb_playlists\\Year Top'
DRY_RUN = False
VERBOSE = True
SLEEP_TIME = 2


for filename in os.listdir(directory):
  filepath = os.path.join(directory, filename)

  if not os.path.isfile(filepath) or not filename.endswith("top.m3u"):
      continue
  try:
    year = filename.split(' top')[0]
    print(100*'*'+f"Found playlist for year: {year}")
  except ValueError:
      print(f"Invalid year format in filename: {filename}")

  # Create or update year like playlist
  pl_name = f'y {year} top'
  # Y.playlist_from_tsv(
  #   tsv_pathfilepath, sort_by_index=True, ignore_banned=True, pl_name=pl_name, 
  #   sleep=SLEEP_TIME, public='PUBLIC', dry=DRY_RUN
  # )
  m3u = frozenset(pd.read_csv(filepath, index_col=None, sep='\t',  header=None)[0].tolist())
  vids = []
  for path in m3u:
    if path not in mb_path_map: continue
    fname = path.split('\\')[-3:]
    vid = mb_path_map[path]
    vids.append(vid)

    
  print(f'[{year}] {len(vids)/len(m3u):.0%} found Musicbee m3u has {len(m3u)} unique tracks, found {len(vids)} ytmusic vids')
  desc = f'Top tracks for {year} (updated {DATE})'
  pl_id = Y.playlist_from_yt_vids(vids, pl_name=pl_name, sleep=3, public='PRIVATE', desc=desc,
                          dry=DRY_RUN, set_rating='LIKE', remove_dupes=True, verbose=VERBOSE)
  

Found playlist for year: 1930s
****************************************************************************************************
[1930s] 79% found Musicbee m3u has 68 unique tracks, found 54 ytmusic vids

Generating y 1930s top ytmusic playlist for 52 tracks
Saved y 1930s top playlist with 52 tracks and playlist id: PLWptjpDqazOwpPEgmbR4H1vMIPe_yndTf, waiting 3 seconds...
Playlist y 1930s top: Found 52 tracks to check for duplicates
Playlist y 1930s top: Found 52 tracks to rate as LIKE
Playlist y 1930s top: Rated 0 of 52 tracks as LIKE
Found playlist for year: 1940s
****************************************************************************************************
[1940s] 90% found Musicbee m3u has 39 unique tracks, found 35 ytmusic vids

Generating y 1940s top ytmusic playlist for 33 tracks
Updated y 1940s top playlist with 33 tracks, playlist id: PLWptjpDqazOwhL4qSOYk8nWrfxiWL5MpW...waiting 3 seconds...
Playlist y 1940s top: Found 85 tracks to check for duplicates
Playlist y 1940